In [1]:
# Importing torch
import torch
import torchvision
import torchmetrics # torch, torchvision, torchaudio and torchmetrics must be downloaded on the same GPU compatability → cu126

import torchsummary
import torchinfo

print(torch.__version__, torchvision.__version__, torchmetrics.__version__, torchinfo.__version__, sep="\n")
print(torchsummary.__name__) # does not have __version__ method

2.14.0+cu126
0.29.0+cu126
1.9.0
1.8.0
torchsummary


In [2]:
# GPU avaliability
if torch.cuda.is_available():
    print("GPU is available!")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"cuda version: {torch.version.cuda}")
else:
    print("GPU not available. Using CPU.")

!nvidia-smi # cuda configurations

GPU is available!
Using GPU: NVIDIA GeForce MX350
cuda version: 12.6
Sat Oct  3 11:15:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce MX350           Off |   00000000:01:00.0 Off |                  N/A |
| N/A   48C    P8            N/A  /   20W |       4MiB /   2048MiB |      0%      Default |
|                                         |                        |   

---

In [3]:
# This method will assign a memory to create a tensor of provided shape and return the garbage/existing numbers at that location
a: torch.Tensor = torch.empty(
    size=[2, 3],
    dtype=torch.float32,
    out=torch.ones(size=[2, 3], dtype=torch.float32, device="cuda"), # shape, dtype & device of both the tensors must be same
    device="cuda",
    pin_memory=True,
    requires_grad=False # operations that use the out parameter are generally non-differentiable in terms of the autograd graph (i.e., they break the computation graph and will typically throw an error if requires_grad=True is involved or cause issues during backpropagation).
)

print(a)

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')


The `out` parameter allows you to specify a pre-allocated tensor where the result of an operation will be written, rather than creating a brand new tensor in memory. <br>

It prevents PyTorch from allocating new memory for the output. This is especially useful in loops, continuous inference pipelines, or memory-constrained environments to avoid garbage collection overhead.

Scaler (0D) → Vector (1D) → Tensor (> 1D)

In [4]:
# Check type
type(a)

torch.Tensor

In [5]:
# Check the memory consuption
from sys import getsizeof

print(f"Total Memory {getsizeof(a)}")
print(f"Tensor Memory {a.nbytes}") # Float of 4 bytes

# Total Memory → 6(total numbers) * [4(size of float32) + 8(size of pointer)] = 72

Total Memory 72
Tensor Memory 24


The core difference is that `sys.getsizeof()` measures the total memory footprint of an object (including overhead and references), while `.nbytes` measures only the raw data size of the object's elements, excluding overhead.

In [6]:
# Using zeros
torch.zeros(
    size = [2, 3], 
    dtype = torch.float32, 
    device="cpu",
    layout=torch.strided, # how a tensor's data is structured and stored in memory. For dense tensors → torch.strided & for sparse tensors → torch.sparse_csr
    pin_memory=True, 
    requires_grad=False
)

tensor([[0., 0., 0.],
        [0., 0., 0.]])

In [7]:
# using ones
torch.ones(size = [2, 3])

tensor([[1., 1., 1.],
        [1., 1., 1.]])

In [8]:
# using rand
torch.rand(size = [2, 3])

tensor([[0.1728, 0.3226, 0.1441],
        [0.0722, 0.0370, 0.6995]])

In [9]:
import torch

# Consider this notebook cell as a Module
seed: int = 42 # NOTE: Store this seed in "params.yaml" and use it throughout the project

# Create a generator object on specified device a pass the seed 
rand_obj = torch.Generator(device="cuda").manual_seed(seed) 
# NOTE: Specifying device is important since memories are different so initializations are also different

# Create a tensor and pass the generator object
tensor: torch.Tensor = torch.rand(size=[2, 3], device="cuda", generator=rand_obj) # NOTE: The device of generator obj and tensor must be same
tensor

tensor([[0.6130, 0.0101, 0.3984],
        [0.0403, 0.1563, 0.4825]], device='cuda:0')

In [10]:
# Working of seed in-order to generate random numbers
g1 = torch.Generator().manual_seed(42)
g2 = torch.Generator().manual_seed(42)

a1 = torch.rand(2, 3, generator=g1)   # first draw from g1
a2 = torch.rand(2, 3, generator=g2)   # first draw from g2
print(torch.equal(a1, a2))                    # TRUE: same stream position

b1 = torch.rand(2, 3, generator=g1)   # second draw from g1
b2 = torch.rand(2, 3, generator=g2)   # second draw from g2
print(torch.equal(b1, b2))                    # TRUE: both at position 2

# But the first and second draws differ:
print(torch.equal(a1, b1))                    # FALSE: different memory positions

True
True
False


In [11]:
# Device agnostic code
from typing import Literal
device: Literal['cpu', 'cuda'] = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [12]:
# using tensor
tensor: torch.Tensor = torch.tensor(
    data = [[1,2,3],[4,5,6]], # list or tuple
    dtype = torch.float32,
    requires_grad = False,
    device = "cpu",
    pin_memory = True # only if tensor is on CPU and want to transfer it on GPU later
); tensor.to(device=device); print(tensor)

# from numpy arrays
import numpy as np

array: np.ndarray = np.array(object=(1, 2, 3))
tensor_from_numpy: torch.Tensor = torch.from_numpy(array)
print(tensor_from_numpy)

tensor([[1., 2., 3.],
        [4., 5., 6.]])
tensor([1, 2, 3])


Mentioning `device → cuda` as well as `pin_memory → True` is a bug

In [13]:
# Question: Which one is fast?
torch.rand(size=[2, 3]).to(device="cuda") # Slow: CPU (pager memory) → RAM (page-locaked memory) → VRAM(GPU memory)
torch.rand(size=[2, 3], pin_memory=True).to(device="cuda") # Itermediate: RAM (page-locaked memory) → VRAM(GPU memory)
torch.rand(size=[2, 3], device="cuda") # Fast: VRAM(GPU memory)

tensor([[0.8582, 0.6359, 0.9329],
        [0.8002, 0.1471, 0.6205]], device='cuda:0')

[A guide on good usage of non_blocking and pin_memory() in PyTorch](https://docs.pytorch.org/tutorials/intermediate/pinmem_nonblock.html)

In [14]:
# NOTE: You can only calculate the gradient when the end result (loss function) is scaler.
try:
    a = torch.rand(size=[2, 3], requires_grad=True, pin_memory=True).to(device=device)
    b: torch.Tensor = a.pow(exponent=2)
    c: torch.Tensor = b * 2 # c = (a ^ 2) + 2

    c.backward()

except RuntimeError as e:
    print(e)

grad can be implicitly created only for scalar outputs


In [15]:
# arange
print("using arange ->", torch.arange(start=0, end=10, step=2))

# using linspace
print("using linspace ->", torch.linspace(start=0, end=10, steps=10))

# using eye - Identity Matrix tensor
print("using eye ->", torch.eye(n=5))

# using full - torch.ones(size = (m, n)) * constant( = 5)
print("using full ->", torch.full(size=(3, 3), fill_value=5))
print(f"using fill -> {torch.fill(input=torch.ones(size=[2, 3]), value=3)}") # replace all the numbers with constant value

using arange -> tensor([0, 2, 4, 6, 8])
using linspace -> tensor([ 0.0000,  1.1111,  2.2222,  3.3333,  4.4444,  5.5556,  6.6667,  7.7778,
         8.8889, 10.0000])
using eye -> tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]])
using full -> tensor([[5, 5, 5],
        [5, 5, 5],
        [5, 5, 5]])
using fill -> tensor([[3., 3., 3.],
        [3., 3., 3.]])


In [16]:
torch.logspace(start=0, end=10, steps=6, base=2, dtype=torch.int32)
# It first creates an tensor containing numbers equidistance from each other like `linspace` -> {0, 2, 4, 6, 8, 10} then the log of what numbers that will give us this perticular tensor -> {log2(1), log2(4)...} 

tensor([   1,    4,   16,   64,  256, 1024], dtype=torch.int32)

---

In [17]:
x: torch.Tensor = torch.tensor([[1,2,3],[4,5,6]])
x

tensor([[1, 2, 3],
        [4, 5, 6]])

In [18]:
print(x.shape)
print(x.size(dim=1)) # You can pass the dimension as well → dim = 1 (rows - outermost dimension)

torch.Size([2, 3])
3


numpy/pandas `axis` paremeter and torch `dim` paremater are same → 0 (innermost), 1(second innermost) ...

In [19]:
# Create an uninitialized tensor with the same shape as the input tensor
y: torch.Tensor = torch.empty_like(
    input=x,
    memory_format=torch.preserve_format, # the way how this tensor(y) get stored in memory is exactly same as the tensor that you are using to create(x) → how tensor get stored in the memory
    dtype=torch.float32,
    layout=torch.strided, # dense tensor
    device=device,
    pin_memory=False,
    requires_grad=True
)

print(y)

tensor([[-0.0000, 1.8396, 0.0000],
        [1.7840, 0.0000, 1.8582]], device='cuda:0', requires_grad=True)


In [20]:
a = torch.rand(size=[5], requires_grad=True)
b = torch.ones_like(input=a, memory_format=torch.preserve_format)
print(b.requires_grad) # no gradient computations by default

print(id(a[1]), id(b[1]))
print(a[1].item(), b[1].item()) 
# temporial behavour of python's id() function for numpy arrays and pytorch tensors explination → /Notes/id method & Torch Memory Format

b[0] = 32
print(a[0].item())

False
134686803840208 134686803840208
0.5589148998260498 1.0
0.6329222917556763


In [21]:
# torch.preserve_format
print(id(x[0][0]))
print(id(y[0][0]))

134686803968592
134686803971024


In [22]:
# more explaination → /Notes/id method & Torch Memory Format
image: torch.Tensor = torch.rand(size=[10, 3, 5, 5])
print(f"memory format before → {image.stride()}")

image = image.to(dtype=torch.float32, device="cuda", memory_format=torch.channels_last)
"""
    torch.channels_last - NHWC [red_pix_0, green_pix_0, blue_pix_0, red_pix_1...]
    torch.contiguous_format - NCHW [red_pix_0, red_pix_1, .... , green_pix_0, ...]
"""
print(f"memory format after → {image.stride()}")

if image.is_contiguous(memory_format=torch.contiguous_format):
    print("image stored by {batch_size, channels, height, width}")
else:
    print("image stored by {batch_size, height, width, channels}")

memory format before → (75, 25, 5, 1)
memory format after → (75, 1, 15, 3)
image stored by {batch_size, height, width, channels}


In [23]:
tensor = torch.empty(size=[2, 3], dtype=torch.float32, device=device, requires_grad=True)
tensor_like: torch.Tensor = torch.empty_like(input=tensor)

print(f"shape: {tensor.shape, tensor_like.shape}")
print(f"dtype: {tensor.dtype, tensor_like.dtype}")
print(f"device: {tensor.device, tensor_like.device}")

try:
    print(f"gradients: {tensor.requires_grad, tensor_like.required_grad}")
except Exception as e:
    print("tensor_like -", e)

shape: (torch.Size([2, 3]), torch.Size([2, 3]))
dtype: (torch.float32, torch.float32)
device: (device(type='cuda', index=0), device(type='cuda', index=0))
tensor_like - 'Tensor' object has no attribute 'required_grad'


In [24]:
torch.ones_like(x)

tensor([[1, 1, 1],
        [1, 1, 1]])

In [25]:
torch.rand_like(x, dtype=torch.float32)

tensor([[0.4769, 0.2201, 0.9366],
        [0.1791, 0.3024, 0.6664]])

`_like()` These methods create a new tensor with the exact same shape, dtype, and device as an existing input tensor but not `gradient computations`.

---

In [26]:
# find data type
x.dtype

torch.int64

In [27]:
# assign data type
torch.tensor([1.0, 2.0, 3.0], dtype = torch.int32)

tensor([1, 2, 3], dtype=torch.int32)

In [28]:
torch.tensor([1, 2, 3], dtype = torch.float64)

tensor([1., 2., 3.], dtype=torch.float64)

In [29]:
# using to() - type conversion, shifting from CPU → GPU, changing memory format
x.to(torch.float32)

tensor([[1., 2., 3.],
        [4., 5., 6.]])

| **Data Type**             | **Dtype**         | **Description**                                                                                                                                                                |
|---------------------------|-------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **32-bit Floating Point** | `torch.float32`   | Standard floating-point type used for most deep learning tasks. Provides a balance between precision and memory usage.                                                         |
| **64-bit Floating Point** | `torch.float64`   | Double-precision floating point. Useful for high-precision numerical tasks but uses more memory.                                                                               |
| **16-bit Floating Point** | `torch.float16`   | Half-precision floating point. Commonly used in mixed-precision training to reduce memory and computational overhead on modern GPUs.                                            |
| **BFloat16**              | `torch.bfloat16`  | Brain floating-point format with reduced precision compared to `float16`. Used in mixed-precision training, especially on TPUs.                                                |
| **8-bit Floating Point**  | `torch.float8`    | Ultra-low-precision floating point. Used for experimental applications and extreme memory-constrained environments (less common).                                               |
| **8-bit Integer**         | `torch.int8`      | 8-bit signed integer. Used for quantized models to save memory and computation in inference.                                                                                   |
| **16-bit Integer**        | `torch.int16`     | 16-bit signed integer. Useful for special numerical tasks requiring intermediate precision.                                                                                    |
| **32-bit Integer**        | `torch.int32`     | Standard signed integer type. Commonly used for indexing and general-purpose numerical tasks.                                                                                  |
| **64-bit Integer**        | `torch.int64`     | Long integer type. Often used for large indexing arrays or for tasks involving large numbers.                                                                                  |
| **8-bit Unsigned Integer**| `torch.uint8`     | 8-bit unsigned integer. Commonly used for image data (e.g., pixel values between 0 and 255).                                                                                    |
| **Boolean**               | `torch.bool`      | Boolean type, stores `True` or `False` values. Often used for masks in logical operations.                                                                                      |
| **Complex 64**            | `torch.complex64` | Complex number type with 32-bit real and 32-bit imaginary parts. Used for scientific and signal processing tasks.                                                               |
| **Complex 128**           | `torch.complex128`| Complex number type with 64-bit real and 64-bit imaginary parts. Offers higher precision but uses more memory.                                                                 |
| **Quantized Integer**     | `torch.qint8`     | Quantized signed 8-bit integer. Used in quantized models for efficient inference.                                                                                              |
| **Quantized Unsigned Integer** | `torch.quint8` | Quantized unsigned 8-bit integer. Often used for quantized tensors in image-related tasks.                                                                                     |


---

#### Mathematical Operations

In [30]:
x = torch.rand(2,2)
x

tensor([[0.9568, 0.2112],
        [0.1020, 0.8286]])

In [31]:
# Scalar Operations

# addition
print(x + 2, end = "\n\n")

# substraction
print(x - 2, end = "\n\n")

# multiplication
print(x * 3, end = "\n\n")

# division
print(x / 3, end = "\n\n")

# int division
print((x * 100) // 3, end = "\n\n")

# mod
print(((x * 100) // 3) % 2, end = "\n\n")

# power
print(x ** 2)

tensor([[2.9568, 2.2112],
        [2.1020, 2.8286]])

tensor([[-1.0432, -1.7888],
        [-1.8980, -1.1714]])

tensor([[2.8705, 0.6336],
        [0.3059, 2.4857]])

tensor([[0.3189, 0.0704],
        [0.0340, 0.2762]])

tensor([[31.,  7.],
        [ 3., 27.]])

tensor([[1., 1.],
        [1., 1.]])

tensor([[0.9155, 0.0446],
        [0.0104, 0.6865]])


In [32]:
# floor (closest smaller int value) division
(torch.tensor([1.0, 2.0, 3.0]) * 100) // 3

tensor([ 33.,  66., 100.])

In [33]:
# Element wise operation
a = torch.rand(2,3)
b = torch.rand(2,3)

print(a)
print(b)

tensor([[0.4148, 0.8699, 0.9744],
        [0.6106, 0.5562, 0.5167]])
tensor([[0.2814, 0.7218, 0.7268],
        [0.6988, 0.7325, 0.4629]])


In [34]:
# Shape must be same or able to broadcast

# add
print(a + b, end="\n\n")

# sub
print(a - b, end="\n\n")

# multiply
print(a * b, end="\n\n")

# division
print(a / b, end="\n\n")

# power
print(a ** b, end="\n\n")

# mod
print(a % b)

tensor([[0.6962, 1.5917, 1.7012],
        [1.3093, 1.2887, 0.9796]])

tensor([[ 0.1334,  0.1481,  0.2476],
        [-0.0882, -0.1763,  0.0537]])

tensor([[0.1167, 0.6279, 0.7082],
        [0.4266, 0.4074, 0.2392]])

tensor([[1.4742, 1.2052, 1.3406],
        [0.8738, 0.7593, 1.1161]])

tensor([[0.7807, 0.9043, 0.9813],
        [0.7084, 0.6507, 0.7366]])

tensor([[0.1334, 0.1481, 0.2476],
        [0.6106, 0.5562, 0.0537]])


In [35]:
c = torch.tensor([1, -2, 3, -4])
c

tensor([ 1, -2,  3, -4])

In [36]:
# abs
torch.abs(c)

tensor([1, 2, 3, 4])

In [37]:
# negative
torch.neg(c)

tensor([-1,  2, -3,  4])

In [38]:
# Inplace operations
print(torch.abs_(c))
print(c)

print(torch.neg_(c))
print(c)

tensor([1, 2, 3, 4])
tensor([1, 2, 3, 4])
tensor([-1, -2, -3, -4])
tensor([-1, -2, -3, -4])


In [39]:
c.fill_(value=10)
# In-place equivalent → c = torch.full_like(c, fill_value=10)

tensor([10, 10, 10, 10])

In [40]:
x = torch.empty_like(input=c)

c.copy_(other=x) # c gets updated, In-place equivalent → c = x.clone()
cloned_c: torch.Tensor = x.clone()

print((c == cloned_c).all().item())

True


In [41]:
d: torch.Tensor = torch.tensor([1.9, 2.3, 3.7, 4.4, 6.5])
d

tensor([1.9000, 2.3000, 3.7000, 4.4000, 6.5000])

In [42]:
# round
torch.round(d) # 6.5 -> 6 (get floored not ceiled)

tensor([2., 2., 4., 4., 6.])

In [43]:
# ceil - nearest bigger integer value
torch.ceil(d)

tensor([2., 3., 4., 5., 7.])

In [44]:
# floor - nearest smaller integer value
torch.floor(d)

tensor([1., 2., 3., 4., 6.])

In [45]:
# clamp | clip - Clipping the values in a certing range
torch.clip(d, min = -1, max = 1)
torch.clamp(d, min = -1, max = 1)

tensor([1., 1., 1., 1., 1.])

In [46]:
e: torch.Tensor = torch.randint(size=(2,3), low=0, high=10, dtype=torch.float32)
e

tensor([[2., 4., 2.],
        [0., 2., 4.]])

In [47]:
# NumPy `axis` vs Pandas `axis` vs PyTorch `dim` → All are same or might change with function
import numpy as np
import pandas as pd
import torch

print(np.array(
    [[1, 2, 3], [4, 5, 6]]
).sum(axis=0))

print(pd.DataFrame(
    [[1, 2, 3], [4, 5, 6]]
).sum(axis=0).to_numpy()) # string arguments → `index` = 0 and `columns` = 1

print(torch.tensor(
    [[1, 2, 3], [4, 5, 6]]
).sum(dim=0))

[5 7 9]
[5 7 9]
tensor([5, 7, 9])


In [48]:
# Aggregate operation

# sum
print(torch.sum(e))

# sum along columns
print(torch.sum(e, dim = 0, keepdim=False)) # same as `axis` parameter in numpy

# sum along rows
print(torch.sum(e, dim = 1, keepdim=False))

# keepdim = True
print(torch.sum(e, dim = 1, keepdim=True))

tensor(14.)
tensor([2., 6., 6.])
tensor([8., 6.])
tensor([[8.],
        [6.]])


If `keepdim` is `False`, the output tensor will not retain the dimension across which the sum was performed.

In [49]:
# mean
torch.mean(e)

# mean along col
torch.mean(e, dim=0)

tensor([1., 3., 3.])

In [50]:
# median
torch.median(e)

tensor(2.)

In [51]:
# max and min
torch.max(e)
torch.min(e)

tensor(0.)

In [52]:
# product
torch.prod(e) # .mul() is a element-wise multiplication between two matrics

tensor(0.)

In [53]:
# standard deviation
torch.std(e)

tensor(1.5055)

In [54]:
# variance
torch.var(e)

tensor(2.2667)

In [55]:
# argmax
torch.argmax(e)

tensor(1)

In [56]:
# argmin
torch.argmin(e)

tensor(3)

In [57]:
# More _methods()
a = torch.full_like(input=e, fill_value=10)
b = a.clone().copy_(a) * 100 # the shape of a and b must be same for copy_ method
print(a, b, sep="\n")

# clone and copy_ → both the tensor elements are pointing to the same memory location

a.clone().add_(b) # a = a + b (ignore clone)
a.clone().sub_(b) # a = a - b 
a.clone().mul_(b) # a = a * b (element wise)

a.zero_() # x = torch.zeros_like(x)

tensor([[10., 10., 10.],
        [10., 10., 10.]])
tensor([[1000., 1000., 1000.],
        [1000., 1000., 1000.]])


tensor([[0., 0., 0.],
        [0., 0., 0.]])

---

#### Matrix operations

In [58]:
f: torch.Tensor = torch.randint(size=(2,3), low=0, high=10)
g: torch.Tensor = torch.randint(size=(3,2), low=0, high=10)

print(f)
print(g)

tensor([[8, 9, 4],
        [8, 7, 6]])
tensor([[4, 5],
        [9, 6],
        [1, 5]])


In [59]:
# matrix multiplcation - dot product
torch.matmul(f, g)

tensor([[117, 114],
        [101, 112]])

In [60]:
vector1: torch.Tensor = torch.tensor([1, 2])
vector2: torch.Tensor = torch.tensor([3, 4])

# dot product - only works with Vectors
torch.dot(vector1, vector2)

tensor(11)

In [61]:
# transpose
torch.transpose(input=f, dim0=0, dim1=1)

tensor([[8, 8],
        [9, 7],
        [4, 6]])

In [62]:
torch.transpose(f, dim0 = 1, dim1 = 0) # Transpose is same as `reshape` operation by swapping the dimensions

tensor([[8, 8],
        [9, 7],
        [4, 6]])

In [63]:
# 0th axis - 1, 1st axis - 2, 2nd axis - 3 → consider size as a list and axis is an index
torch.rand(size=[1, 2, 3,]).transpose(dim0=0, dim1=2).shape

torch.Size([3, 2, 1])

In [64]:
# Implace Transpose → only works with 2D
f.t_()

tensor([[8, 8],
        [9, 7],
        [4, 6]])

In [65]:
temp: torch.Tensor = torch.tensor([
    [[1], [2]],
    [[1], [3]],
    [[1], [4]],
])

print(temp.shape)
print(torch.transpose(temp, dim0=0, dim1=1).shape) # Check the dimensions and its index correctly

torch.Size([3, 2, 1])
torch.Size([2, 3, 1])


Tip: To visualize a transpose, flatten the tensor into a 1D array and then re-arrange the elements according to the target shape.
<br>For a tensor with shape [2, 3, 1], the 0th dimension has size 2, the 1st dimension has size 3, and so on.

In [66]:
# Convert any nD tensor to 1D tensor
print(torch.tensor([[1, 2], [3, 4]]).ravel())
print(torch.tensor([[1, 2], [3, 4]]).flatten())
print(torch.tensor([[[1], [2]], [[3], [4]]]).reshape(-1))

tensor([1, 2, 3, 4])
tensor([1, 2, 3, 4])
tensor([1, 2, 3, 4])


In [67]:
h: torch.Tensor = torch.randint(size=(3,3), low=0, high=10, dtype=torch.float32)
h

tensor([[0., 1., 1.],
        [9., 7., 4.],
        [7., 7., 0.]])

In [68]:
# determinant
torch.det(h)

tensor(42.)

In [69]:
# inverse
torch.inverse(h)

tensor([[-0.6667,  0.1667, -0.0714],
        [ 0.6667, -0.1667,  0.2143],
        [ 0.3333,  0.1667, -0.2143]])

---

#### Comparison operations

In [70]:
i: torch.Tensor = torch.randint(size=(2,3), low=0, high=10)
j: torch.Tensor = torch.randint(size=(2,3), low=0, high=10)

print(i)
print(j)

tensor([[0, 1, 9],
        [7, 0, 8]])
tensor([[2, 1, 0],
        [6, 0, 4]])


In [71]:
# Element wise comparison - Shape must be same or able to broadcast

# greater than
print(i > j)

# less than
print(i < j)

# equal to
print(i == j)

# not equal to
print(i != j)

tensor([[False, False,  True],
        [ True, False,  True]])
tensor([[ True, False, False],
        [False, False, False]])
tensor([[False,  True, False],
        [False,  True, False]])
tensor([[ True, False,  True],
        [ True, False,  True]])


In [72]:
# Boolean Indexing - Reshaped into 1D
i[i > j]

tensor([9, 7, 8])

In [73]:
# Compare the shape & elements
torch.equal(
    torch.tensor([2, 3], device="cuda"),
    torch.tensor([2, 3], device="cuda")
)

True

---

#### Special functions

In [74]:
k: torch.Tensor = torch.randint(size=(2,3), low=0, high=10, dtype=torch.float64)
k

tensor([[1., 4., 5.],
        [6., 0., 6.]], dtype=torch.float64)

In [75]:
# log - base e
torch.log(input = k)

# log - base 10
torch.log10(input = k)

tensor([[0.0000, 0.6021, 0.6990],
        [0.7782,   -inf, 0.7782]], dtype=torch.float64)

In [76]:
# Checking the size of the elements in the tensor
tensor = torch.log10(input = k) # implicit type conversion from float32 to float 64
print(tensor.dtype)
print(tensor.element_size())  # size in bytes of a single element

torch.float64
8


In [77]:
# exp
torch.exp(k)

tensor([[  2.7183,  54.5982, 148.4132],
        [403.4288,   1.0000, 403.4288]], dtype=torch.float64)

In [78]:
# sqrt
torch.sqrt(k)

tensor([[1.0000, 2.0000, 2.2361],
        [2.4495, 0.0000, 2.4495]], dtype=torch.float64)

In [79]:
# sigmoid
torch.sigmoid(k)

tensor([[0.7311, 0.9820, 0.9933],
        [0.9975, 0.5000, 0.9975]], dtype=torch.float64)

In [80]:
# softmax
torch.softmax(k, dim = 1)

tensor([[0.0132, 0.2654, 0.7214],
        [0.4994, 0.0012, 0.4994]], dtype=torch.float64)

In [81]:
# relu
torch.relu(k)

tensor([[1., 4., 5.],
        [6., 0., 6.]], dtype=torch.float64)

In [82]:
# tile - Constructs a tensor by repeating the elements of input tensor across dimentions
tile_tensor: torch.Tensor = torch.tensor([[
    [1, 2, 3],
    [4, 5, 6]
]])
torch.tile(
    input=tile_tensor,
    dims=[2, 3] # change this value to [3], [2, 3], [2, 3, 4] to see how tile function reperation varies
).shape

torch.Size([1, 4, 9])

---

#### Inplace Operations

In [83]:
m: torch.Tensor = torch.rand(2,3)
n: torch.Tensor = torch.rand(2,3)

print(m)
print(n)

tensor([[0.1875, 0.7767, 0.1951],
        [0.4384, 0.4873, 0.5399]])
tensor([[8.2433e-04, 8.4887e-01, 1.0922e-01],
        [2.1231e-01, 3.2572e-01, 8.3308e-01]])


In [84]:
# _ methods
m.add_(n) # It returns the new matrix

tensor([[0.1883, 1.6256, 0.3043],
        [0.6507, 0.8131, 1.3730]])

In [85]:
m # Updated

tensor([[0.1883, 1.6256, 0.3043],
        [0.6507, 0.8131, 1.3730]])

In [86]:
n

tensor([[8.2433e-04, 8.4887e-01, 1.0922e-01],
        [2.1231e-01, 3.2572e-01, 8.3308e-01]])

In [87]:
torch.relu(m)

tensor([[0.1883, 1.6256, 0.3043],
        [0.6507, 0.8131, 1.3730]])

In [88]:
m.relu_()

tensor([[0.1883, 1.6256, 0.3043],
        [0.6507, 0.8131, 1.3730]])

In [89]:
m

tensor([[0.1883, 1.6256, 0.3043],
        [0.6507, 0.8131, 1.3730]])

---

#### Copying a Tensor

In [90]:
a = torch.tensor(
    data = [[1,2,3], [4,5,6]]
)
b = a.clone()

print(a, b, sep = '\n')

print(id(a), id(b))
print(id(a[0,0]), id(b[0][0]))

tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([[1, 2, 3],
        [4, 5, 6]])
134686199802576 134686199792976
134686199799376 134686199804368


In [91]:
a[0][0] = 10

print(a, b, sep = '\n')

tensor([[10,  2,  3],
        [ 4,  5,  6]])
tensor([[1, 2, 3],
        [4, 5, 6]])


---

A third-order polynomial, trained to predict `y = sin(x) from -pi to pi` by minimizing squared Euclidean distance.

In [92]:
# Variables Initialization
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype: torch._C.dtype = torch.float
learning_rate = 1e-6

# Dataset
x = torch.linspace(start=-torch.pi, end=torch.pi, steps=2000) # shape -> 2000 1D
y = torch.sin(input=x) # Shape -> 2000 1D

# Random Weights → for 3 degree term | shape -> Scaler
a = torch.randn((), dtype=dtype)
b = torch.randn((), dtype=dtype)
c = torch.randn((), dtype=dtype)
d = torch.randn((), dtype=dtype)

def y_pred_equation(X_train):
    y_pred: torch.Tensor = a + (b * x) + (c * x.pow(2)) + (d * x.pow(3))
    return y_pred # shape -> 2000 1D

def main():
    # Training
    for i in range(2500):
        # Prediction
        y_pred = y_pred_equation(x)

        # Loss Calculation
        loss = (y - y_pred).pow(2).sum().item()

        if i % 100 == 99:
            print(f"loss at {i + 1}it epoch → {loss}")

        # Gradients Calculation
        grad_y_pred = -2 * (y - y_pred) # shape -> 2000
        grad_a: torch.Tensor = grad_y_pred.sum()
        grad_b: torch.Tensor = torch.matmul(input = grad_y_pred, other = x).sum()
        grad_c: torch.Tensor = torch.matmul(input = grad_y_pred, other = x.pow(2)).sum()
        grad_d: torch.Tensor = torch.matmul(input = grad_y_pred, other = x.pow(3)).sum()

        # Updating
        global a, b, c, d
        a = a - learning_rate * grad_a
        b -= learning_rate * grad_b
        c -= learning_rate * grad_c
        d -= learning_rate * grad_d

main()

loss at 100it epoch → 141.9485321044922
loss at 200it epoch → 97.32842254638672
loss at 300it epoch → 67.68382263183594
loss at 400it epoch → 47.98222732543945
loss at 500it epoch → 34.88444900512695
loss at 600it epoch → 26.17405128479004
loss at 700it epoch → 20.379314422607422
loss at 800it epoch → 16.522672653198242
loss at 900it epoch → 13.95496940612793
loss at 1000it epoch → 12.244643211364746
loss at 1100it epoch → 11.104866027832031
loss at 1200it epoch → 10.344980239868164
loss at 1300it epoch → 9.838094711303711
loss at 1400it epoch → 9.499802589416504
loss at 1500it epoch → 9.273902893066406
loss at 1600it epoch → 9.122964859008789
loss at 1700it epoch → 9.022040367126465
loss at 1800it epoch → 8.954530715942383
loss at 1900it epoch → 8.909337997436523
loss at 2000it epoch → 8.879066467285156
loss at 2100it epoch → 8.858769416809082
loss at 2200it epoch → 8.845151901245117
loss at 2300it epoch → 8.836008071899414
loss at 2400it epoch → 8.829862594604492
loss at 2500it epoch